In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(r'C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_ERA5Land_Monthly_SeasonFiltered.csv')

In [5]:
df = df.dropna() 

In [6]:
df

,lat0,lon0,year,month,tmin_C,tmax_C,vpd,duration,intensity,frequency
1,-31.75,19.00,1980,1,10.830026,40.145288,1.602520,3.0,0.889349,1.0
2,-31.50,18.25,1980,1,13.238229,37.348825,0.782230,5.0,0.632091,1.0
3,-31.50,18.75,1980,1,10.718698,41.689310,1.439980,3.0,0.663272,1.0
4,-31.50,19.00,1980,1,10.503854,40.657007,1.588102,3.0,0.667949,1.0
5,-31.25,18.00,1980,1,12.691400,39.409219,0.881258,3.0,0.587841,1.0
...,...,...,...,...,...,...,...,...,...,...
692275,6.75,14.25,2024,12,16.998895,32.284799,2.250072,3.0,0.412221,2.0
692276,6.75,14.50,2024,12,16.586816,31.870737,2.151149,3.0,0.182525,2.0
692277,7.00,12.75,2024,12,16.506738,33.780115,2.151773,3.0,0.296496,2.0
692278,7.00,14.00,2024,12,16.209833,30.927484,2.027194,3.0,0.355153,2.0


In [5]:
import numpy as np
import pandas as pd

# -------------------------
# INPUT / OUTPUT PATHS
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_ERA5Land_Monthly_SeasonFiltered.csv"

# (change these if you want)
out_delta_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000.csv"
out_rich_csv  = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_means_and_deltas_2000.csv"

# -------------------------
# SETTINGS
# -------------------------
POINT_COLS = ["lat0", "lon0"]     # point identifier
YEAR_COL = "year"
MONTH_COL = "month"
SPLIT_YEAR = 2000                 # before < 2000 ; after >= 2000

# -------------------------
# LOAD
# -------------------------
df = pd.read_csv(in_csv)

# Basic checks
missing = [c for c in POINT_COLS + [YEAR_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# -------------------------
# PICK VALUE COLUMNS (numeric vars except point/time columns)
# -------------------------
exclude = set(POINT_COLS + [YEAR_COL, MONTH_COL])
value_cols = [
    c for c in df.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
]
if not value_cols:
    raise ValueError("No numeric value columns found to compute deltas for.")

# -------------------------
# FLAG PERIODS
# -------------------------
df["period"] = np.where(df[YEAR_COL] < SPLIT_YEAR, "before_2000", "after_2000")

# -------------------------
# MEANS BY POINT + PERIOD
# -------------------------
means = (
    df.groupby(POINT_COLS + ["period"], dropna=False)[value_cols]
      .mean(numeric_only=True)          # ignores NaNs by default
      .reset_index()
)

# Pivot to wide format: one row per point, columns per period
wide = means.pivot(index=POINT_COLS, columns="period", values=value_cols)

# Flatten the MultiIndex columns -> "var__before_2000", "var__after_2000"
wide.columns = [f"{var}__{period}" for (var, period) in wide.columns]
wide = wide.reset_index()

# -------------------------
# DELTAS (after - before)
# -------------------------
for var in value_cols:
    wide[f"{var}__delta_after_minus_before"] = (
        wide.get(f"{var}__after_2000") - wide.get(f"{var}__before_2000")
    )

# Deltas-only table
delta_cols = [f"{v}__delta_after_minus_before" for v in value_cols]
out_delta = wide[POINT_COLS + delta_cols].copy()

# Rich table (before/after means + deltas)
out_rich = wide.copy()

# -------------------------
# SAVE
# -------------------------
out_delta.to_csv(out_delta_csv, index=False)
out_rich.to_csv(out_rich_csv, index=False)

print("Saved:")
print("  Deltas only:", out_delta_csv)
print("  Before/After means + deltas:", out_rich_csv)
print(f"Points: {len(out_delta):,}")
print("Value columns used:", value_cols)


Saved:
  Deltas only: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000.csv
  Before/After means + deltas: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_means_and_deltas_2000.csv
Points: 5,128
Value columns used: ['tmin_C', 'tmax_C', 'vpd', 'duration', 'intensity', 'frequency']


In [4]:
import numpy as np
import pandas as pd

# -------------------------
# INPUT / OUTPUT PATHS
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_ERA5Land_Monthly_SeasonFiltered.csv"

# (change these if you want)
out_delta_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_MAX_after_minus_before_2000.csv"
out_rich_csv  = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_MAX_and_deltas_2000.csv"

# -------------------------
# SETTINGS
# -------------------------
POINT_COLS = ["lat0", "lon0"]
YEAR_COL = "year"
MONTH_COL = "month"
SPLIT_YEAR = 2000  # before < 2000 ; after >= 2000

# -------------------------
# LOAD
# -------------------------
df = pd.read_csv(in_csv)

# Basic checks
required = POINT_COLS + [YEAR_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# -------------------------
# VALUE COLUMNS (numeric vars except point/time columns)
# -------------------------
exclude = set(POINT_COLS + [YEAR_COL, MONTH_COL])
value_cols = [
    c for c in df.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
]
if not value_cols:
    raise ValueError("No numeric value columns found to compute deltas for.")

# -------------------------
# PERIOD FLAG
# -------------------------
df["period"] = np.where(df[YEAR_COL] < SPLIT_YEAR, "before_2000", "after_2000")

# -------------------------
# MAX BY POINT + PERIOD
# -------------------------
maxvals = (
    df.groupby(POINT_COLS + ["period"], dropna=False)[value_cols]
      .max(numeric_only=True)   # ignores NaNs by default
      .reset_index()
)

# Pivot wide: one row per point
wide = maxvals.pivot(index=POINT_COLS, columns="period", values=value_cols)

# Flatten columns -> "var__before_2000_max", "var__after_2000_max"
wide.columns = [f"{var}__{period}_max" for (var, period) in wide.columns]
wide = wide.reset_index()

# -------------------------
# DELTAS (after_max - before_max)
# -------------------------
for var in value_cols:
    wide[f"{var}__delta_after_minus_before_max"] = (
        wide.get(f"{var}__after_2000_max") - wide.get(f"{var}__before_2000_max")
    )

# Deltas-only output
delta_cols = [f"{v}__delta_after_minus_before_max" for v in value_cols]
out_delta = wide[POINT_COLS + delta_cols].copy()

# Rich output (before/after max + deltas)
out_rich = wide.copy()

# -------------------------
# SAVE
# -------------------------
out_delta.to_csv(out_delta_csv, index=False)
out_rich.to_csv(out_rich_csv, index=False)

print("Saved:")
print("  Deltas only:", out_delta_csv)
print("  Before/After MAX + deltas:", out_rich_csv)
print(f"Points: {len(out_delta):,}")
print("Value columns used:", value_cols)


Saved:
  Deltas only: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_MAX_after_minus_before_2000.csv
  Before/After MAX + deltas: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_MAX_and_deltas_2000.csv
Points: 5,128
Value columns used: ['tmin_C', 'tmax_C', 'vpd', 'duration', 'intensity', 'frequency']


In [6]:
import numpy as np
import pandas as pd

# -------------------------
# INPUT / OUTPUT PATHS
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_ERA5Land_Monthly_SeasonFiltered.csv"

out_delta_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000.csv"
out_rich_csv  = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_means_and_deltas_2000.csv"

# -------------------------
# SETTINGS
# -------------------------
POINT_COLS = ["lat0", "lon0"]
YEAR_COL = "year"
MONTH_COL = "month"
SPLIT_YEAR = 2000  # before < 2000 ; after >= 2000

# -------------------------
# LOAD
# -------------------------
df = pd.read_csv(in_csv)

missing = [c for c in POINT_COLS + [YEAR_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# -------------------------
# PICK VALUE COLUMNS
# -------------------------
exclude = set(POINT_COLS + [YEAR_COL, MONTH_COL])
value_cols = [
    c for c in df.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
]
if not value_cols:
    raise ValueError("No numeric value columns found to compute deltas for.")

# -------------------------
# FLAG PERIODS
# -------------------------
df["period"] = np.where(df[YEAR_COL] < SPLIT_YEAR, "before_2000", "after_2000")

# -------------------------
# MEANS BY POINT + PERIOD
# -------------------------
means = (
    df.groupby(POINT_COLS + ["period"], dropna=False)[value_cols]
      .mean(numeric_only=True)
      .reset_index()
)

wide = means.pivot(index=POINT_COLS, columns="period", values=value_cols)
wide.columns = [f"{var}__{period}" for (var, period) in wide.columns]
wide = wide.reset_index()

# -------------------------
# DELTAS (after - before)
# -------------------------
for var in value_cols:
    wide[f"{var}__delta_after_minus_before"] = (
        wide.get(f"{var}__after_2000") - wide.get(f"{var}__before_2000")
    )

delta_cols = [f"{v}__delta_after_minus_before" for v in value_cols]

# -------------------------
# REMOVE ROWS WHERE ANY DELTA IS NEGATIVE
# (NaNs are kept; only strictly negative triggers removal)
# -------------------------
neg_any = (wide[delta_cols] < 0).any(axis=1)
wide = wide.loc[~neg_any].copy()

# -------------------------
# OUTPUT TABLES
# -------------------------
out_delta = wide[POINT_COLS + delta_cols].copy()
out_rich  = wide.copy()

# -------------------------
# SAVE
# -------------------------
out_delta.to_csv(out_delta_csv, index=False)
out_rich.to_csv(out_rich_csv, index=False)

print("Saved:")
print("  Deltas only:", out_delta_csv)
print("  Before/After means + deltas:", out_rich_csv)
print(f"Points kept (no negative deltas): {len(out_delta):,}")
print("Value columns used:", value_cols)


Saved:
  Deltas only: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000.csv
  Before/After means + deltas: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_means_and_deltas_2000.csv
Points kept (no negative deltas): 1,518
Value columns used: ['tmin_C', 'tmax_C', 'vpd', 'duration', 'intensity', 'frequency']


In [9]:
import pandas as pd

df = pd.read_csv(r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000.csv")

# grab all delta columns
delta_cols = [c for c in df.columns if c.endswith("__delta_after_minus_before")]

# Pearson correlation matrix (r)
R = df[delta_cols].corr(method="pearson")

print(R.round(3))

# optional: save
R.to_csv(r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\pearson_r_matrix_all_deltas.csv")


                                     tmin_C__delta_after_minus_before  \
tmin_C__delta_after_minus_before                                1.000   
tmax_C__delta_after_minus_before                                0.226   
vpd__delta_after_minus_before                                   0.271   
duration__delta_after_minus_before                             -0.042   
intensity__delta_after_minus_before                             0.239   
frequency__delta_after_minus_before                            -0.068   

                                     tmax_C__delta_after_minus_before  \
tmin_C__delta_after_minus_before                                0.226   
tmax_C__delta_after_minus_before                                1.000   
vpd__delta_after_minus_before                                   0.580   
duration__delta_after_minus_before                              0.071   
intensity__delta_after_minus_before                             0.186   
frequency__delta_after_minus_before               

In [3]:
import os
import pandas as pd

folder = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data"

master_file = "Final_ERA5Land_Monthly_SeasonFiltered.csv"

wind_file  = "ERA5Land_Monthly_Wind10_u10_v10_wind10_Points_1980_present.csv"
swvl1_file = "ERA5Land_Monthly_SWVL1_Points_1980_present.csv"
swvl2_file = "ERA5Land_Monthly_SWVL2_Points_1980_present.csv"
swvl3_file = "ERA5Land_Monthly_SWVL3_Points_1980_present.csv"
swvl4_file = "ERA5Land_Monthly_SWVL4_Points_1980_present.csv"

KEYS = ["lat0", "lon0", "year", "month"]

def prep_keys(df, round_dp=2):
    df = df.copy()
    df["lat0"] = pd.to_numeric(df["lat0"], errors="coerce").round(round_dp)
    df["lon0"] = pd.to_numeric(df["lon0"], errors="coerce").round(round_dp)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["month"] = pd.to_numeric(df["month"], errors="coerce").astype("Int64")
    return df.dropna(subset=KEYS)

def collapse_duplicates(df):
    # After rounding, duplicates can happen → average numeric columns
    return df.groupby(KEYS, as_index=False).mean(numeric_only=True)

def find_value_col(df):
    # pick first non-key column if name is unknown
    non_keys = [c for c in df.columns if c not in KEYS]
    if not non_keys:
        raise ValueError("No value column found in SWVL file.")
    return non_keys[0]

# -------------------
# 1) Load master keys
# -------------------
master_path = os.path.join(folder, master_file)
master = pd.read_csv(master_path)
master = prep_keys(master, round_dp=2)

# keep a de-duplicated key table for matching
master_keys = master[KEYS].drop_duplicates()
print("Master rows:", len(master), "Unique keys:", len(master_keys))

# -------------------
# 2) Load WIND vars
# -------------------
wind_path = os.path.join(folder, wind_file)
wind = pd.read_csv(wind_path)
wind = prep_keys(wind, round_dp=2)
wind = wind[KEYS + [c for c in ["u10","v10","wind10"] if c in wind.columns]]
wind = collapse_duplicates(wind)

# keep only rows present in master
wind = master_keys.merge(wind, on=KEYS, how="left")

# -------------------
# 3) Load SWVL vars
# -------------------
def load_swvl(swvl_path, new_name):
    df = pd.read_csv(swvl_path)
    df = prep_keys(df, round_dp=2)
    val_col = find_value_col(df)
    df = df[KEYS + [val_col]].rename(columns={val_col: new_name})
    df = collapse_duplicates(df)
    # filter to master keys
    return master_keys.merge(df, on=KEYS, how="left")

swvl1 = load_swvl(os.path.join(folder, swvl1_file), "swvl1")
swvl2 = load_swvl(os.path.join(folder, swvl2_file), "swvl2")
swvl3 = load_swvl(os.path.join(folder, swvl3_file), "swvl3")
swvl4 = load_swvl(os.path.join(folder, swvl4_file), "swvl4")

# -------------------
# 4) Merge into master
# -------------------
out = master.merge(wind,  on=KEYS, how="left")
out = out.merge(swvl1, on=KEYS, how="left")
out = out.merge(swvl2, on=KEYS, how="left")
out = out.merge(swvl3, on=KEYS, how="left")
out = out.merge(swvl4, on=KEYS, how="left")

# -------------------
# 5) Save
# -------------------
out_path = os.path.join(folder, "Final_SeasonFiltered_with_ERA5vars.csv")
out.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Final columns:", out.columns.tolist())
print("Final shape:", out.shape)


Master rows: 706665 Unique keys: 706665
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_SeasonFiltered_with_ERA5vars.csv
Final columns: ['lat0', 'lon0', 'year', 'month', 'tmin_C', 'tmax_C', 'vpd', 'duration', 'intensity', 'frequency', 'u10', 'v10', 'wind10']
Final shape: (706665, 13)


In [9]:
import pandas as pd

# -------------------------
# FULL PATHS (explicit)
# -------------------------
master_path = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_ERA5Land_Monthly_SeasonFiltered.csv"

wind_path  = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\ERA5Land_Monthly_Wind10_u10_v10_wind10_Points_1980_present.csv"
swvl1_path = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\ERA5Land_Monthly_SWVL1_Points_1980_present.csv"
swvl2_path = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\ERA5Land_Monthly_SWVL2_Points_1980_present.csv"
swvl3_path = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\ERA5Land_Monthly_SWVL3_Points_1980_present.csv"
swvl4_path = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\ERA5Land_Monthly_SWVL4_Points_1980_present.csv"

out_path   = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_SeasonFiltered_with_ERA5vars.csv"

KEYS = ["lat0", "lon0", "year", "month"]

def prep_keys(df, round_dp=2):
    df = df.copy()
    df["lat0"] = pd.to_numeric(df["lat0"], errors="coerce").round(round_dp)
    df["lon0"] = pd.to_numeric(df["lon0"], errors="coerce").round(round_dp)
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["month"] = pd.to_numeric(df["month"], errors="coerce").astype("Int64")
    return df.dropna(subset=KEYS)

def collapse_duplicates(df):
    # After rounding, duplicates can happen → average numeric columns
    return df.groupby(KEYS, as_index=False).mean(numeric_only=True)

def find_value_col(df):
    # pick first non-key column if name is unknown
    non_keys = [c for c in df.columns if c not in KEYS]
    if not non_keys:
        raise ValueError("No value column found in SWVL file.")
    return non_keys[0]

# -------------------
# 1) Load master keys
# -------------------
master = pd.read_csv(master_path)
master = prep_keys(master, round_dp=2)

master_keys = master[KEYS].drop_duplicates()
print("Master rows:", len(master), "Unique keys:", len(master_keys))

# -------------------
# 2) Load WIND vars
# -------------------
wind = pd.read_csv(wind_path)
wind = prep_keys(wind, round_dp=2)
wind = wind[KEYS + [c for c in ["wind10"] if c in wind.columns]]
wind = collapse_duplicates(wind)

wind = master_keys.merge(wind, on=KEYS, how="left")

# -------------------
# 3) Load SWVL vars
# -------------------
def load_swvl(path, new_name):
    df = pd.read_csv(path)
    df = prep_keys(df, round_dp=2)

    # ✅ pick the correct value column
    if new_name in df.columns:
        val_col = new_name
    else:
        # fallback: any column that contains 'swvl'
        swvl_cols = [c for c in df.columns if "swvl" in c.lower()]
        if not swvl_cols:
            raise ValueError(f"No SWVL column found in {path}. Columns: {df.columns.tolist()}")
        val_col = swvl_cols[0]

    # make sure numeric
    df[val_col] = pd.to_numeric(df[val_col], errors="coerce")

    df = df[KEYS + [val_col]].rename(columns={val_col: new_name})

    # average duplicates only for that column
    df = df.groupby(KEYS, as_index=False)[new_name].mean()

    return master_keys.merge(df, on=KEYS, how="left")


swvl1 = load_swvl(swvl1_path, "swvl1")
swvl2 = load_swvl(swvl2_path, "swvl2")
swvl3 = load_swvl(swvl3_path, "swvl3")
swvl4 = load_swvl(swvl4_path, "swvl4")

# -------------------
# 4) Merge into master
# -------------------
out = master.merge(wind,  on=KEYS, how="left")
out = out.merge(swvl1, on=KEYS, how="left")
out = out.merge(swvl2, on=KEYS, how="left")
out = out.merge(swvl3, on=KEYS, how="left")
out = out.merge(swvl4, on=KEYS, how="left")

# -------------------
# 5) Save
# -------------------
out.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Final columns:", out.columns.tolist())
print("Final shape:", out.shape)


Master rows: 706665 Unique keys: 706665
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_SeasonFiltered_with_ERA5vars.csv
Final columns: ['lat0', 'lon0', 'year', 'month', 'tmin_C', 'tmax_C', 'vpd', 'duration', 'intensity', 'frequency', 'wind10', 'swvl1', 'swvl2', 'swvl3', 'swvl4']
Final shape: (706665, 15)


In [10]:
import numpy as np
import pandas as pd

# -------------------------
# INPUT / OUTPUT PATHS
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_SeasonFiltered_with_ERA5vars.csv"

out_delta_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"
out_rich_csv  = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_means_and_deltas_2000_new.csv"

# -------------------------
# SETTINGS
# -------------------------
POINT_COLS = ["lat0", "lon0"]
YEAR_COL = "year"
MONTH_COL = "month"
SPLIT_YEAR = 2000  # before < 2000 ; after >= 2000

# -------------------------
# LOAD
# -------------------------
df = pd.read_csv(in_csv)

missing = [c for c in POINT_COLS + [YEAR_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# -------------------------
# PICK VALUE COLUMNS
# -------------------------
exclude = set(POINT_COLS + [YEAR_COL, MONTH_COL])
value_cols = [
    c for c in df.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
]
if not value_cols:
    raise ValueError("No numeric value columns found to compute deltas for.")

# -------------------------
# FLAG PERIODS
# -------------------------
df["period"] = np.where(df[YEAR_COL] < SPLIT_YEAR, "before_2000", "after_2000")

# -------------------------
# MEANS BY POINT + PERIOD
# -------------------------
means = (
    df.groupby(POINT_COLS + ["period"], dropna=False)[value_cols]
      .mean(numeric_only=True)
      .reset_index()
)

wide = means.pivot(index=POINT_COLS, columns="period", values=value_cols)
wide.columns = [f"{var}__{period}" for (var, period) in wide.columns]
wide = wide.reset_index()

# -------------------------
# DELTAS (after - before)
# -------------------------
for var in value_cols:
    wide[f"{var}__delta_after_minus_before"] = (
        wide.get(f"{var}__after_2000") - wide.get(f"{var}__before_2000")
    )

delta_cols = [f"{v}__delta_after_minus_before" for v in value_cols]

# -------------------------
# REMOVE ROWS WHERE ANY DELTA IS NEGATIVE
# (NaNs are kept; only strictly negative triggers removal)
# -------------------------
neg_any = (wide[delta_cols] < 0).any(axis=1)
wide = wide.loc[~neg_any].copy()

# -------------------------
# OUTPUT TABLES
# -------------------------
out_delta = wide[POINT_COLS + delta_cols].copy()
out_rich  = wide.copy()

# -------------------------
# SAVE
# -------------------------
out_delta.to_csv(out_delta_csv, index=False)
out_rich.to_csv(out_rich_csv, index=False)

print("Saved:")
print("  Deltas only:", out_delta_csv)
print("  Before/After means + deltas:", out_rich_csv)
print(f"Points kept (no negative deltas): {len(out_delta):,}")
print("Value columns used:", value_cols)


Saved:
  Deltas only: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv
  Before/After means + deltas: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_means_and_deltas_2000_new.csv
Points kept (no negative deltas): 19
Value columns used: ['tmin_C', 'tmax_C', 'vpd', 'duration', 'intensity', 'frequency', 'wind10', 'swvl1', 'swvl2', 'swvl3', 'swvl4']


In [11]:
import numpy as np
import pandas as pd

# -------------------------
# INPUT / OUTPUT PATHS
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\Final_SeasonFiltered_with_ERA5vars.csv"

out_delta_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"
out_rich_csv  = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_means_and_deltas_2000_new.csv"

# -------------------------
# SETTINGS
# -------------------------
POINT_COLS = ["lat0", "lon0"]
YEAR_COL = "year"
MONTH_COL = "month"
SPLIT_YEAR = 2000  # before < 2000 ; after >= 2000

# Only these types of variables must have NON-NEGATIVE deltas (after - before)
POS_KEYWORDS = ["intensity", "freq", "frequency", "dur", "duration"]

# -------------------------
# LOAD
# -------------------------
df = pd.read_csv(in_csv)

missing = [c for c in POINT_COLS + [YEAR_COL] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# -------------------------
# PICK VALUE COLUMNS
# -------------------------
exclude = set(POINT_COLS + [YEAR_COL, MONTH_COL])
value_cols = [
    c for c in df.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
]
if not value_cols:
    raise ValueError("No numeric value columns found to compute deltas for.")

# -------------------------
# FLAG PERIODS
# -------------------------
df["period"] = np.where(df[YEAR_COL] < SPLIT_YEAR, "before_2000", "after_2000")

# -------------------------
# MEANS BY POINT + PERIOD
# -------------------------
means = (
    df.groupby(POINT_COLS + ["period"], dropna=False)[value_cols]
      .mean(numeric_only=True)
      .reset_index()
)

wide = means.pivot(index=POINT_COLS, columns="period", values=value_cols)
wide.columns = [f"{var}__{period}" for (var, period) in wide.columns]
wide = wide.reset_index()

# -------------------------
# DELTAS (after - before)
# -------------------------
for var in value_cols:
    wide[f"{var}__delta_after_minus_before"] = (
        wide.get(f"{var}__after_2000") - wide.get(f"{var}__before_2000")
    )

delta_cols = [f"{v}__delta_after_minus_before" for v in value_cols]

# -------------------------
# FILTER: NEGATIVES ONLY FOR intensity/freq/dur VARIABLES
# (Other variables are allowed to be negative)
# NaNs are kept; only strictly negative triggers removal.
# -------------------------
pos_delta_cols = [
    c for c in delta_cols
    if any(k in c.lower() for k in POS_KEYWORDS)
]

print("Value columns used:", value_cols)
print("Delta columns constrained to be non-negative:", pos_delta_cols)

if pos_delta_cols:
    neg_any = (wide[pos_delta_cols] < 0).any(axis=1)
    wide = wide.loc[~neg_any].copy()
else:
    print("WARNING: No intensity/freq/dur delta columns found. No filtering applied.")

# -------------------------
# OUTPUT TABLES
# -------------------------
out_delta = wide[POINT_COLS + delta_cols].copy()
out_rich  = wide.copy()

# -------------------------
# SAVE
# -------------------------
out_delta.to_csv(out_delta_csv, index=False)
out_rich.to_csv(out_rich_csv, index=False)

print("Saved:")
print("  Deltas only:", out_delta_csv)
print("  Before/After means + deltas:", out_rich_csv)
print(f"Points kept (after filtering): {len(out_delta):,}")


Value columns used: ['tmin_C', 'tmax_C', 'vpd', 'duration', 'intensity', 'frequency', 'wind10', 'swvl1', 'swvl2', 'swvl3', 'swvl4']
Delta columns constrained to be non-negative: ['duration__delta_after_minus_before', 'intensity__delta_after_minus_before', 'frequency__delta_after_minus_before']
Saved:
  Deltas only: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv
  Before/After means + deltas: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_before_after_means_and_deltas_2000_new.csv
Points kept (after filtering): 1,556


In [13]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# INPUT
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"

out_dir = os.path.dirname(in_csv)
out_corr_csv = os.path.join(out_dir, "pearson_r_target_vs_others_freq_dur_intensity.csv")
out_heatmap_png = os.path.join(out_dir, "pearson_r_target_vs_others_freq_dur_intensity.png")

# -------------------------
# LOAD
# -------------------------
df = pd.read_csv(in_csv)

# Targets (rows)
targets = [
    "frequency__delta_after_minus_before",
    "duration__delta_after_minus_before",
    "intensity__delta_after_minus_before",
]

# Others (columns)
others = [
    "tmin_C__delta_after_minus_before",
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "wind10__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
    "swvl3__delta_after_minus_before",
    "swvl4__delta_after_minus_before",
]

needed = targets + others
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

# Ensure numeric
X = df[needed].apply(pd.to_numeric, errors="coerce")

# -------------------------
# PEARSON r: targets vs others
# -------------------------
corr_mat = pd.DataFrame(index=targets, columns=others, dtype=float)

for t in targets:
    for o in others:
        corr_mat.loc[t, o] = X[[t, o]].corr(method="pearson", min_periods=2).iloc[0, 1]

# Save table
corr_mat.to_csv(out_corr_csv)
print("Saved correlation table:", out_corr_csv)

# -------------------------
# HEATMAP
# -------------------------
plt.figure(figsize=(12, 4))
plt.imshow(corr_mat.values, aspect="auto")
plt.colorbar(label="Pearson r")
plt.xticks(range(len(others)), others, rotation=45, ha="right")
plt.yticks(range(len(targets)), targets)
plt.title("Pearson r (freq/dur/intensity) vs other deltas")
plt.tight_layout()
plt.savefig(out_heatmap_png, dpi=300)
plt.close()

print("Saved heatmap:", out_heatmap_png)
print("Done. Table shape:", corr_mat.shape)


Saved correlation table: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\pearson_r_target_vs_others_freq_dur_intensity.csv
Saved heatmap: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\pearson_r_target_vs_others_freq_dur_intensity.png
Done. Table shape: (3, 8)


In [14]:
import numpy as np
import pandas as pd

targets = ["frequency__delta_after_minus_before",
           "duration__delta_after_minus_before",
           "intensity__delta_after_minus_before"]

others = ["tmin_C__delta_after_minus_before",
          "tmax_C__delta_after_minus_before",
          "vpd__delta_after_minus_before",
          "wind10__delta_after_minus_before",
          "swvl1__delta_after_minus_before",
          "swvl2__delta_after_minus_before",
          "swvl3__delta_after_minus_before",
          "swvl4__delta_after_minus_before"]

df = pd.read_csv(r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv")
X = df[targets + others].apply(pd.to_numeric, errors="coerce")

R = pd.DataFrame(index=targets, columns=others, dtype=float)
N = pd.DataFrame(index=targets, columns=others, dtype=int)

for t in targets:
    for o in others:
        pair = X[[t, o]].dropna()
        N.loc[t, o] = len(pair)
        R.loc[t, o] = pair[t].corr(pair[o]) if len(pair) >= 2 else np.nan

print("R (Pearson):\n", R)
print("\nN used:\n", N)


R (Pearson):
                                      tmin_C__delta_after_minus_before  \
frequency__delta_after_minus_before                         -0.055806   
duration__delta_after_minus_before                          -0.019889   
intensity__delta_after_minus_before                          0.233325   

                                     tmax_C__delta_after_minus_before  \
frequency__delta_after_minus_before                          0.501226   
duration__delta_after_minus_before                           0.085792   
intensity__delta_after_minus_before                          0.186196   

                                     vpd__delta_after_minus_before  \
frequency__delta_after_minus_before                       0.525784   
duration__delta_after_minus_before                        0.084641   
intensity__delta_after_minus_before                       0.138215   

                                     wind10__delta_after_minus_before  \
frequency__delta_after_minus_before           

In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# INPUT
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"
out_dir = os.path.dirname(in_csv)
os.makedirs(out_dir, exist_ok=True)

# -------------------------
# VARIABLES
# -------------------------
targets = [
    "frequency__delta_after_minus_before",
    "duration__delta_after_minus_before",
    "intensity__delta_after_minus_before",
]

drivers = [
    "tmin_C__delta_after_minus_before",
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "wind10__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
    "swvl3__delta_after_minus_before",
    "swvl4__delta_after_minus_before",
]

# -------------------------
# HELPERS
# -------------------------
def linreg_ci_band(x, y, x_grid=None, alpha=0.05):
    """
    OLS y = b0 + b1*x, and 95% CI for the MEAN prediction (not prediction interval).
    Returns: x_grid, y_hat, y_lo, y_hi, b0, b1, r, n
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    n = len(x)
    if n < 3:
        raise ValueError("Need at least 3 points for CI band.")

    xbar = x.mean()
    ybar = y.mean()

    Sxx = np.sum((x - xbar) ** 2)
    Sxy = np.sum((x - xbar) * (y - ybar))

    b1 = Sxy / Sxx
    b0 = ybar - b1 * xbar

    yhat = b0 + b1 * x
    resid = y - yhat
    s2 = np.sum(resid ** 2) / (n - 2)          # residual variance
    s = np.sqrt(s2)

    # t critical value for 95% CI
    # (no scipy) -> approximate using normal for n>=30, otherwise a small lookup fallback
    # For typical point counts, n is usually large; normal approx is fine.
    # z=1.96 for 95%
    tcrit = 1.96 if n >= 30 else 2.0

    if x_grid is None:
        x_grid = np.linspace(np.nanmin(x), np.nanmax(x), 200)

    x_grid = np.asarray(x_grid, dtype=float)
    y_grid = b0 + b1 * x_grid

    # Standard error of mean prediction at x0:
    # se_mean(x0) = s * sqrt( 1/n + (x0 - xbar)^2 / Sxx )
    se_mean = s * np.sqrt(1.0 / n + (x_grid - xbar) ** 2 / Sxx)

    y_lo = y_grid - tcrit * se_mean
    y_hi = y_grid + tcrit * se_mean

    # Pearson r
    r = np.corrcoef(x, y)[0, 1]

    return x_grid, y_grid, y_lo, y_hi, b0, b1, r, n


def make_reg_plot(df, y_col, x_col, out_png):
    # pairwise dropna
    pair = df[[x_col, y_col]].apply(pd.to_numeric, errors="coerce").dropna()
    if len(pair) < 3:
        print(f"SKIP (too few points): {y_col} vs {x_col} (n={len(pair)})")
        return

    x = pair[x_col].values
    y = pair[y_col].values

    xg, yg, lo, hi, b0, b1, r, n = linreg_ci_band(x, y)

    plt.figure(figsize=(7, 5))
    plt.scatter(x, y)                # no fixed colors (default)
    plt.plot(xg, yg)                 # regression line
    plt.fill_between(xg, lo, hi, alpha=0.2)  # 95% CI band (mean)

    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f"{y_col} vs {x_col}\nOLS: y={b0:.3g}+{b1:.3g}x, r={r:.3f}, n={n}")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()
    print("Saved:", out_png)


# -------------------------
# RUN
# -------------------------
df = pd.read_csv(in_csv)

for y_col in targets:
    for x_col in drivers:
        safe_y = y_col.replace("__delta_after_minus_before", "").replace("__", "_")
        safe_x = x_col.replace("__delta_after_minus_before", "").replace("__", "_")
        out_png = os.path.join(out_dir, f"reg_{safe_y}__vs__{safe_x}_95CI.png")
        make_reg_plot(df, y_col, x_col, out_png)

print("Done.")


Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_frequency__vs__tmin_C_95CI.png
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_frequency__vs__tmax_C_95CI.png
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_frequency__vs__vpd_95CI.png
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_frequency__vs__wind10_95CI.png
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_frequency__vs__swvl1_95CI.png
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_frequency__vs__swvl2_95CI.png
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_frequency__vs__swvl3_95CI.png
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_frequency__vs__swvl4_95CI.png
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_duration__vs__tmin_C_95CI.png
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\reg_duration__vs__tmax_C_95CI.png

In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# INPUT
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"
out_dir = os.path.dirname(in_csv)
os.makedirs(out_dir, exist_ok=True)

out_png = os.path.join(out_dir, "regression_matrix_freq_dur_intensity_vs_drivers_95CI.png")

# -------------------------
# VARIABLES
# -------------------------
targets = [
    "frequency__delta_after_minus_before",
    "duration__delta_after_minus_before",
    "intensity__delta_after_minus_before",
]

drivers = [
    "tmin_C__delta_after_minus_before",
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "wind10__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
    "swvl3__delta_after_minus_before",
    "swvl4__delta_after_minus_before",
]

# Short labels (for cleaner axes)
short = {
    "frequency__delta_after_minus_before": "Freq Δ",
    "duration__delta_after_minus_before": "Dur Δ",
    "intensity__delta_after_minus_before": "Inten Δ",
    "tmin_C__delta_after_minus_before": "Tmin Δ",
    "tmax_C__delta_after_minus_before": "Tmax Δ",
    "vpd__delta_after_minus_before": "VPD Δ",
    "wind10__delta_after_minus_before": "Wind Δ",
    "swvl1__delta_after_minus_before": "SWVL1 Δ",
    "swvl2__delta_after_minus_before": "SWVL2 Δ",
    "swvl3__delta_after_minus_before": "SWVL3 Δ",
    "swvl4__delta_after_minus_before": "SWVL4 Δ",
}

# -------------------------
# HELPERS
# -------------------------
def linreg_ci_band(x, y, x_grid=None):
    """
    OLS y = b0 + b1*x, and 95% CI for the MEAN prediction.
    Returns: x_grid, y_hat, y_lo, y_hi, r, n
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    n = len(x)
    if n < 3:
        raise ValueError("Need at least 3 points for CI band.")

    xbar = x.mean()
    ybar = y.mean()

    Sxx = np.sum((x - xbar) ** 2)
    if Sxx == 0:
        raise ValueError("All x values identical (Sxx=0).")

    Sxy = np.sum((x - xbar) * (y - ybar))
    b1 = Sxy / Sxx
    b0 = ybar - b1 * xbar

    yhat = b0 + b1 * x
    resid = y - yhat
    s2 = np.sum(resid ** 2) / (n - 2)
    s = np.sqrt(s2)

    # 95% critical value (normal approx; good when n is not tiny)
    tcrit = 1.96 if n >= 30 else 2.0

    if x_grid is None:
        x_grid = np.linspace(np.nanmin(x), np.nanmax(x), 200)

    x_grid = np.asarray(x_grid, dtype=float)
    y_grid = b0 + b1 * x_grid

    se_mean = s * np.sqrt(1.0 / n + (x_grid - xbar) ** 2 / Sxx)
    y_lo = y_grid - tcrit * se_mean
    y_hi = y_grid + tcrit * se_mean

    r = np.corrcoef(x, y)[0, 1]
    return x_grid, y_grid, y_lo, y_hi, r, n


# -------------------------
# RUN
# -------------------------
df = pd.read_csv(in_csv)

# ensure all needed columns exist
needed = targets + drivers
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

# numeric conversion once
X = df[needed].apply(pd.to_numeric, errors="coerce")

nrows, ncols = len(targets), len(drivers)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 2.8*nrows), squeeze=False)

for i, y_col in enumerate(targets):
    for j, x_col in enumerate(drivers):
        ax = axes[i, j]

        pair = X[[x_col, y_col]].dropna()
        n = len(pair)

        # Scatter
        if n > 0:
            ax.scatter(pair[x_col].values, pair[y_col].values, s=8)

        # Regression + 95% CI (mean)
        if n >= 3:
            try:
                xg, yg, lo, hi, r, n_used = linreg_ci_band(pair[x_col].values, pair[y_col].values)
                ax.plot(xg, yg, linewidth=1.5)
                ax.fill_between(xg, lo, hi, alpha=0.2)
                ax.text(0.02, 0.98, f"r={r:.2f}\nn={n_used}",
                        transform=ax.transAxes, va="top", ha="left", fontsize=9)
            except Exception as e:
                ax.text(0.02, 0.98, f"n={n}\n(no fit)",
                        transform=ax.transAxes, va="top", ha="left", fontsize=9)
        else:
            ax.text(0.02, 0.98, f"n={n}\n(no fit)",
                    transform=ax.transAxes, va="top", ha="left", fontsize=9)

        # Labels (only outer edges to keep it clean)
        if i == nrows - 1:
            ax.set_xlabel(short.get(x_col, x_col), fontsize=10)
        else:
            ax.set_xlabel("")
            ax.set_xticklabels([])

        if j == 0:
            ax.set_ylabel(short.get(y_col, y_col), fontsize=10)
        else:
            ax.set_ylabel("")
            ax.set_yticklabels([])

# Title + spacing
fig.suptitle("Regression matrix (OLS) with 95% CI band (mean): Targets vs Drivers (deltas after - before 2000)",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close()

print("Saved:", out_png)
print("Done.")


Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\regression_matrix_freq_dur_intensity_vs_drivers_95CI.png
Done.


In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -------------------------
# INPUT
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"
out_dir = os.path.dirname(in_csv)
os.makedirs(out_dir, exist_ok=True)

out_png = os.path.join(out_dir, "heatmap_plus_freq_top4_regressions.png")

# -------------------------
# VARIABLES
# -------------------------
targets = [
    "frequency__delta_after_minus_before",
    "duration__delta_after_minus_before",
    "intensity__delta_after_minus_before",
]

drivers = [
    "tmin_C__delta_after_minus_before",
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "wind10__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
    "swvl3__delta_after_minus_before",
    "swvl4__delta_after_minus_before",
]

short = {
    "frequency__delta_after_minus_before": "Freq Δ",
    "duration__delta_after_minus_before": "Dur Δ",
    "intensity__delta_after_minus_before": "Inten Δ",
    "tmin_C__delta_after_minus_before": "Tmin Δ",
    "tmax_C__delta_after_minus_before": "Tmax Δ",
    "vpd__delta_after_minus_before": "VPD Δ",
    "wind10__delta_after_minus_before": "Wind Δ",
    "swvl1__delta_after_minus_before": "SWVL1 Δ",
    "swvl2__delta_after_minus_before": "SWVL2 Δ",
    "swvl3__delta_after_minus_before": "SWVL3 Δ",
    "swvl4__delta_after_minus_before": "SWVL4 Δ",
}

# -------------------------
# YOU CHOOSE THESE 4 (fixed)
# -------------------------
freq_target = "frequency__delta_after_minus_before"
freq_drivers_to_plot = [
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
]

# -------------------------
# HELPERS
# -------------------------
def corr_r(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def linreg_ci_band(x, y, x_grid=None):
    """
    OLS y = b0 + b1*x, and 95% CI for the MEAN prediction.
    Returns x_grid, y_hat, y_lo, y_hi
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 3:
        raise ValueError("Need >=3 points")

    xbar = x.mean()
    ybar = y.mean()
    Sxx = np.sum((x - xbar) ** 2)
    if Sxx == 0:
        raise ValueError("Sxx=0")

    Sxy = np.sum((x - xbar) * (y - ybar))
    b1 = Sxy / Sxx
    b0 = ybar - b1 * xbar

    yhat = b0 + b1 * x
    resid = y - yhat
    s2 = np.sum(resid ** 2) / (n - 2)
    s = np.sqrt(s2)

    tcrit = 1.96 if n >= 30 else 2.0

    if x_grid is None:
        x_grid = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    x_grid = np.asarray(x_grid, dtype=float)

    y_grid = b0 + b1 * x_grid
    se_mean = s * np.sqrt(1.0 / n + (x_grid - xbar) ** 2 / Sxx)
    y_lo = y_grid - tcrit * se_mean
    y_hi = y_grid + tcrit * se_mean
    return x_grid, y_grid, y_lo, y_hi

# -------------------------
# RUN
# -------------------------
df = pd.read_csv(in_csv)

needed = targets + drivers
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

X = df[needed].apply(pd.to_numeric, errors="coerce")

# Build correlation matrix for heatmap
R = np.full((len(targets), len(drivers)), np.nan)
for i, y in enumerate(targets):
    for j, x in enumerate(drivers):
        R[i, j] = corr_r(X[x].values, X[y].values)

# -------------------------
# FIGURE LAYOUT:
# heatmap on top (spans 2 columns)
# 4 regression plots below (2x2)
# -------------------------
fig = plt.figure(figsize=(8, 11))
gs = GridSpec(nrows=3, ncols=2, figure=fig, height_ratios=[1.35, 1.0, 1.0], hspace=0.35, wspace=0.25)

# Heatmap axis spanning top row across both columns
ax_hm = fig.add_subplot(gs[0, :])
im = ax_hm.imshow(R, vmin=-1, vmax=1, aspect="auto")

ax_hm.set_xticks(np.arange(len(drivers)))
ax_hm.set_yticks(np.arange(len(targets)))
ax_hm.set_xticklabels([short.get(d, d) for d in drivers], rotation=30, ha="right")
ax_hm.set_yticklabels([short.get(t, t) for t in targets])

# annotate with r only
for i in range(len(targets)):
    for j in range(len(drivers)):
        r = R[i, j]
        ax_hm.text(j, i, "NA" if not np.isfinite(r) else f"{r:.2f}",
                   ha="center", va="center", fontsize=10)

cbar = fig.colorbar(im, ax=ax_hm, fraction=0.03, pad=0.02)
cbar.set_label("Pearson r")

ax_hm.set_title("Targets vs Drivers (Δ after − before 2000): correlation heatmap")
ax_hm.set_xlabel("Drivers")
ax_hm.set_ylabel("Targets")

# ---- Bottom 4 regression plots (2x2) ----
reg_axes = [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[2, 0]),
    fig.add_subplot(gs[2, 1]),
]

for ax, xcol in zip(reg_axes, freq_drivers_to_plot):
    pair = X[[xcol, freq_target]].dropna()
    x = pair[xcol].values
    y = pair[freq_target].values

    ax.scatter(x, y, s=10, alpha=0.75)

    r = corr_r(x, y)
    if len(pair) >= 3 and np.isfinite(r):
        try:
            xg, yg, lo, hi = linreg_ci_band(x, y)
            ax.plot(xg, yg, linewidth=2)
            ax.fill_between(xg, lo, hi, alpha=0.18)
        except Exception:
            pass

    ax.set_title(f"{short[freq_target]} vs {short.get(xcol, xcol)}  (r={r:.2f})", fontsize=10)
    ax.set_xlabel(short.get(xcol, xcol))
    ax.set_ylabel(short[freq_target])
    ax.grid(True, linewidth=0.4, alpha=0.35)

plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", out_png)

Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\heatmap_plus_freq_top4_regressions.png


In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -------------------------
# INPUT
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"
out_dir = os.path.dirname(in_csv)
os.makedirs(out_dir, exist_ok=True)

out_png = os.path.join(out_dir, "heatmap_plus_freq_top4_regressions.png")

# -------------------------
# VARIABLES
# -------------------------
targets = [
    "frequency__delta_after_minus_before",
    "duration__delta_after_minus_before",
    "intensity__delta_after_minus_before",
]

drivers = [
    "tmin_C__delta_after_minus_before",
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "wind10__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
    "swvl3__delta_after_minus_before",
    "swvl4__delta_after_minus_before",
]

short = {
    "frequency__delta_after_minus_before": "Freq Δ",
    "duration__delta_after_minus_before": "Dur Δ",
    "intensity__delta_after_minus_before": "Inten Δ",
    "tmin_C__delta_after_minus_before": "Tmin Δ",
    "tmax_C__delta_after_minus_before": "Tmax Δ",
    "vpd__delta_after_minus_before": "VPD Δ",
    "wind10__delta_after_minus_before": "Wind Δ",
    "swvl1__delta_after_minus_before": "SWVL1 Δ",
    "swvl2__delta_after_minus_before": "SWVL2 Δ",
    "swvl3__delta_after_minus_before": "SWVL3 Δ",
    "swvl4__delta_after_minus_before": "SWVL4 Δ",
}

# -------------------------
# YOU CHOOSE THESE 4 (fixed)
# -------------------------
freq_target = "frequency__delta_after_minus_before"
freq_drivers_to_plot = [
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
]

# -------------------------
# HELPERS
# -------------------------
def corr_r(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def linreg_ci_band(x, y, x_grid=None):
    """
    OLS y = b0 + b1*x, and 95% CI for the MEAN prediction.
    Returns x_grid, y_hat, y_lo, y_hi
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 3:
        raise ValueError("Need >=3 points")

    xbar = x.mean()
    ybar = y.mean()
    Sxx = np.sum((x - xbar) ** 2)
    if Sxx == 0:
        raise ValueError("Sxx=0")

    Sxy = np.sum((x - xbar) * (y - ybar))
    b1 = Sxy / Sxx
    b0 = ybar - b1 * xbar

    yhat = b0 + b1 * x
    resid = y - yhat
    s2 = np.sum(resid ** 2) / (n - 2)
    s = np.sqrt(s2)

    tcrit = 1.96 if n >= 30 else 2.0

    if x_grid is None:
        x_grid = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    x_grid = np.asarray(x_grid, dtype=float)

    y_grid = b0 + b1 * x_grid
    se_mean = s * np.sqrt(1.0 / n + (x_grid - xbar) ** 2 / Sxx)
    y_lo = y_grid - tcrit * se_mean
    y_hi = y_grid + tcrit * se_mean
    return x_grid, y_grid, y_lo, y_hi

# -------------------------
# RUN
# -------------------------
df = pd.read_csv(in_csv)

needed = targets + drivers
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

X = df[needed].apply(pd.to_numeric, errors="coerce")

# Build correlation matrix for heatmap
R = np.full((len(targets), len(drivers)), np.nan)
for i, y in enumerate(targets):
    for j, x in enumerate(drivers):
        R[i, j] = corr_r(X[x].values, X[y].values)

# -------------------------
# FIGURE LAYOUT:
# heatmap on top (spans 2 columns)
# 4 regression plots below (2x2)
# -------------------------
fig = plt.figure(figsize=(8, 11))
gs = GridSpec(nrows=3, ncols=2, figure=fig, height_ratios=[1.35, 1.0, 1.0], hspace=0.35, wspace=0.25)

# Heatmap axis spanning top row across both columns
ax_hm = fig.add_subplot(gs[0, :])
im = ax_hm.imshow(R, vmin=-1, vmax=1, aspect="auto")

ax_hm.set_xticks(np.arange(len(drivers)))
ax_hm.set_yticks(np.arange(len(targets)))
ax_hm.set_xticklabels([short.get(d, d) for d in drivers], rotation=30, ha="right")
ax_hm.set_yticklabels([short.get(t, t) for t in targets])

# annotate with r only
for i in range(len(targets)):
    for j in range(len(drivers)):
        r = R[i, j]
        ax_hm.text(j, i, "NA" if not np.isfinite(r) else f"{r:.2f}",
                   ha="center", va="center", fontsize=10)

cbar = fig.colorbar(im, ax=ax_hm, fraction=0.03, pad=0.02)
cbar.set_label("Pearson r")

ax_hm.set_title("Targets vs Drivers (Δ after − before 2000): correlation heatmap")
ax_hm.set_xlabel("Drivers")
ax_hm.set_ylabel("Targets")

# ---- Bottom 4 regression plots (2x2) ----
reg_axes = [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[2, 0]),
    fig.add_subplot(gs[2, 1]),
]

for ax, xcol in zip(reg_axes, freq_drivers_to_plot):
    pair = X[[xcol, freq_target]].dropna()
    x = pair[xcol].values
    y = pair[freq_target].values

    ax.scatter(x, y, s=10, alpha=0.75)

    r = corr_r(x, y)
    if len(pair) >= 3 and np.isfinite(r):
        try:
            xg, yg, lo, hi = linreg_ci_band(x, y)
            ax.plot(xg, yg, linewidth=2)
            ax.fill_between(xg, lo, hi, alpha=0.18)
        except Exception:
            pass

    ax.set_title(f"{short[freq_target]} vs {short.get(xcol, xcol)}  (r={r:.2f})", fontsize=10)
    ax.set_xlabel(short.get(xcol, xcol))
    ax.set_ylabel(short[freq_target])
    ax.grid(True, linewidth=0.4, alpha=0.35)

plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", out_png)

Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\heatmap_plus_freq_top4_regressions.png


In [12]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -------------------------
# INPUT
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"
out_dir = os.path.dirname(in_csv)
os.makedirs(out_dir, exist_ok=True)

out_png = os.path.join(out_dir, "heatmap_plus_freq_top4_regressions.png")

# -------------------------
# VARIABLES
# -------------------------
targets = [
    "frequency__delta_after_minus_before",
    "duration__delta_after_minus_before",
    "intensity__delta_after_minus_before",
]

drivers = [
    "tmin_C__delta_after_minus_before",
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "wind10__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
    "swvl3__delta_after_minus_before",
    "swvl4__delta_after_minus_before",
]

# -------------------------
# LABELS (Δ BEFORE the variable)
# -------------------------
short = {
    "frequency__delta_after_minus_before": "Δ Freq",
    "duration__delta_after_minus_before": "Δ Dur",
    "intensity__delta_after_minus_before": "Δ Inten",
    "tmin_C__delta_after_minus_before": "Δ Tmin",
    "tmax_C__delta_after_minus_before": "Δ Tmax",
    "vpd__delta_after_minus_before": "Δ VPD",
    "wind10__delta_after_minus_before": "Δ Wind",
    "swvl1__delta_after_minus_before": "Δ SWVL1",
    "swvl2__delta_after_minus_before": "Δ SWVL2",
    "swvl3__delta_after_minus_before": "Δ SWVL3",
    "swvl4__delta_after_minus_before": "Δ SWVL4",
}

# -------------------------
# YOU CHOOSE THESE 4 (fixed)
# -------------------------
freq_target = "frequency__delta_after_minus_before"
freq_drivers_to_plot = [
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
]

# -------------------------
# HELPERS
# -------------------------
def corr_r(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def linreg_ci_band(x, y, x_grid=None):
    """
    OLS y = b0 + b1*x, and 95% CI for the MEAN prediction.
    Returns x_grid, y_hat, y_lo, y_hi
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 3:
        raise ValueError("Need >=3 points")

    xbar = x.mean()
    ybar = y.mean()
    Sxx = np.sum((x - xbar) ** 2)
    if Sxx == 0:
        raise ValueError("Sxx=0")

    Sxy = np.sum((x - xbar) * (y - ybar))
    b1 = Sxy / Sxx
    b0 = ybar - b1 * xbar

    yhat = b0 + b1 * x
    resid = y - yhat
    s2 = np.sum(resid ** 2) / (n - 2)
    s = np.sqrt(s2)

    tcrit = 1.96 if n >= 30 else 2.0

    if x_grid is None:
        x_grid = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    x_grid = np.asarray(x_grid, dtype=float)

    y_grid = b0 + b1 * x_grid
    se_mean = s * np.sqrt(1.0 / n + (x_grid - xbar) ** 2 / Sxx)
    y_lo = y_grid - tcrit * se_mean
    y_hi = y_grid + tcrit * se_mean
    return x_grid, y_grid, y_lo, y_hi

# -------------------------
# RUN
# -------------------------
df = pd.read_csv(in_csv)

needed = targets + drivers
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

X = df[needed].apply(pd.to_numeric, errors="coerce")

# Build correlation matrix for heatmap
R = np.full((len(targets), len(drivers)), np.nan)
for i, y in enumerate(targets):
    for j, x in enumerate(drivers):
        R[i, j] = corr_r(X[x].values, X[y].values)

# -------------------------
# FIGURE LAYOUT:
# heatmap on top (spans 2 columns)
# 4 regression plots below (2x2)
# -------------------------
fig = plt.figure(figsize=(8, 11))
gs = GridSpec(
    nrows=3, ncols=2, figure=fig,
    height_ratios=[1.35, 1.0, 1.0],
    hspace=0.35, wspace=0.25
)

# Heatmap axis spanning top row across both columns
ax_hm = fig.add_subplot(gs[0, :])
im = ax_hm.imshow(R, vmin=-1, vmax=1, aspect="auto")

ax_hm.set_xticks(np.arange(len(drivers)))
ax_hm.set_yticks(np.arange(len(targets)))
ax_hm.set_xticklabels([short.get(d, d) for d in drivers], rotation=30, ha="right")
ax_hm.set_yticklabels([short.get(t, t) for t in targets])

# annotate with r only
for i in range(len(targets)):
    for j in range(len(drivers)):
        r = R[i, j]
        ax_hm.text(
            j, i,
            "NA" if not np.isfinite(r) else f"{r:.2f}",
            ha="center", va="center", fontsize=10
        )

cbar = fig.colorbar(im, ax=ax_hm, fraction=0.03, pad=0.02)
cbar.set_label("Pearson r")

ax_hm.set_title("Thirstwave Metrics vs Hydroclimate Drivers")
ax_hm.set_xlabel("Drivers")
ax_hm.set_ylabel("Targets")

# ---- Bottom 4 regression plots (2x2) ----
reg_axes = [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
    fig.add_subplot(gs[2, 0]),
    fig.add_subplot(gs[2, 1]),
]

for ax, xcol in zip(reg_axes, freq_drivers_to_plot):
    pair = X[[xcol, freq_target]].dropna()
    x = pair[xcol].values
    y = pair[freq_target].values

    ax.scatter(x, y, s=10, alpha=0.75)

    r = corr_r(x, y)
    if len(pair) >= 3 and np.isfinite(r):
        try:
            xg, yg, lo, hi = linreg_ci_band(x, y)
            ax.plot(xg, yg, linewidth=2)
            ax.fill_between(xg, lo, hi, alpha=0.18)
        except Exception:
            pass

    ax.set_title(f"{short[freq_target]} vs {short.get(xcol, xcol)}  (r={r:.2f})", fontsize=10)
    ax.set_xlabel(short.get(xcol, xcol))
    ax.set_ylabel(short[freq_target])
    ax.grid(True, linewidth=0.4, alpha=0.35)

plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", out_png)


Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\heatmap_plus_freq_top4_regressions.png


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -------------------------
# INPUT
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"
out_dir = os.path.dirname(in_csv)
os.makedirs(out_dir, exist_ok=True)
out_png = os.path.join(out_dir, "top4_each_target_regressions_3cols.png")

# -------------------------
# VARIABLES
# -------------------------
targets = [
    "frequency__delta_after_minus_before",
    "duration__delta_after_minus_before",
    "intensity__delta_after_minus_before",
]

drivers = [
    "tmin_C__delta_after_minus_before",
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "wind10__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
    "swvl3__delta_after_minus_before",
    "swvl4__delta_after_minus_before",
]

# -------------------------
# LABELS
# -------------------------
short = {
    "frequency__delta_after_minus_before": "Δ Freq",
    "duration__delta_after_minus_before": "Δ Dur",
    "intensity__delta_after_minus_before": "Δ Inten",
    "tmin_C__delta_after_minus_before": "Δ Tmin",
    "tmax_C__delta_after_minus_before": "Δ Tmax",
    "vpd__delta_after_minus_before": "Δ VPD",
    "wind10__delta_after_minus_before": "Δ Wind",
    "swvl1__delta_after_minus_before": "Δ SWVL1",
    "swvl2__delta_after_minus_before": "Δ SWVL2",
    "swvl3__delta_after_minus_before": "Δ SWVL3",
    "swvl4__delta_after_minus_before": "Δ SWVL4",
}

TOPK = 4

# -------------------------
# HELPERS
# -------------------------
def corr_r(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def linreg_ci_band(x, y, x_grid=None):
    """
    OLS y = b0 + b1*x, and 95% CI for the MEAN prediction.
    Returns x_grid, y_hat, y_lo, y_hi
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 3:
        raise ValueError("Need >=3 points")

    xbar = x.mean()
    ybar = y.mean()
    Sxx = np.sum((x - xbar) ** 2)
    if Sxx == 0:
        raise ValueError("Sxx=0")

    Sxy = np.sum((x - xbar) * (y - ybar))
    b1 = Sxy / Sxx
    b0 = ybar - b1 * xbar

    yhat = b0 + b1 * x
    resid = y - yhat
    s2 = np.sum(resid ** 2) / (n - 2)
    s = np.sqrt(s2)

    # simple t-crit approximation
    tcrit = 1.96 if n >= 30 else 2.0

    if x_grid is None:
        x_grid = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    x_grid = np.asarray(x_grid, dtype=float)

    y_grid = b0 + b1 * x_grid
    se_mean = s * np.sqrt(1.0 / n + (x_grid - xbar) ** 2 / Sxx)
    y_lo = y_grid - tcrit * se_mean
    y_hi = y_grid + tcrit * se_mean
    return x_grid, y_grid, y_lo, y_hi

# -------------------------
# RUN
# -------------------------
df = pd.read_csv(in_csv)

needed = targets + drivers
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

X = df[needed].apply(pd.to_numeric, errors="coerce")

# Compute correlations and pick top-4 drivers for each target by |r|
top_drivers_by_target = {}
top_r_by_target = {}

for ycol in targets:
    r_list = []
    for xcol in drivers:
        r = corr_r(X[xcol].values, X[ycol].values)
        r_list.append((xcol, r))
    r_list = [(x, r) for x, r in r_list if np.isfinite(r)]
    r_list.sort(key=lambda t: abs(t[1]), reverse=True)
    top_drivers_by_target[ycol] = [x for x, _ in r_list[:TOPK]]
    top_r_by_target[ycol] = {x: r for x, r in r_list[:TOPK]}

# -------------------------
# PLOT: 4 rows x 3 columns
# -------------------------
fig = plt.figure(figsize=(12, 12))
gs = GridSpec(nrows=TOPK, ncols=len(targets), figure=fig, hspace=0.45, wspace=0.30)

for col, ycol in enumerate(targets):
    chosen = top_drivers_by_target.get(ycol, [])
    for row in range(TOPK):
        ax = fig.add_subplot(gs[row, col])

        if row >= len(chosen):
            ax.axis("off")
            continue

        xcol = chosen[row]
        pair = X[[xcol, ycol]].dropna()
        x = pair[xcol].values
        y = pair[ycol].values

        ax.scatter(x, y, s=10, alpha=0.75)

        r = corr_r(x, y)
        if len(pair) >= 3 and np.isfinite(r):
            try:
                xg, yg, lo, hi = linreg_ci_band(x, y)
                ax.plot(xg, yg, linewidth=2)
                ax.fill_between(xg, lo, hi, alpha=0.18)
            except Exception:
                pass

        ax.set_title(f"{short.get(ycol, ycol)} vs {short.get(xcol, xcol)} (r={r:.2f})", fontsize=10)
        ax.set_xlabel(short.get(xcol, xcol))
        ax.set_ylabel(short.get(ycol, ycol))
        ax.grid(True, linewidth=0.4, alpha=0.35)

plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", out_png)


Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\top4_each_target_regressions_3cols.png


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -------------------------
# INPUT
# -------------------------
in_csv = r"C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\point_deltas_after_minus_before_2000_new.csv"
out_dir = os.path.dirname(in_csv)
os.makedirs(out_dir, exist_ok=True)
out_png = os.path.join(out_dir, "top4_freq_drivers_applied_to_all_targets_3cols.png")

# -------------------------
# VARIABLES
# -------------------------
targets = [
    "frequency__delta_after_minus_before",
    "duration__delta_after_minus_before",
    "intensity__delta_after_minus_before",
]

drivers = [
    "tmin_C__delta_after_minus_before",
    "tmax_C__delta_after_minus_before",
    "vpd__delta_after_minus_before",
    "wind10__delta_after_minus_before",
    "swvl1__delta_after_minus_before",
    "swvl2__delta_after_minus_before",
    "swvl3__delta_after_minus_before",
    "swvl4__delta_after_minus_before",
]

short = {
    "frequency__delta_after_minus_before": "Δ Freq",
    "duration__delta_after_minus_before": "Δ Dur",
    "intensity__delta_after_minus_before": "Δ Inten",
    "tmin_C__delta_after_minus_before": "Δ Tmin",
    "tmax_C__delta_after_minus_before": "Δ Tmax",
    "vpd__delta_after_minus_before": "Δ VPD",
    "wind10__delta_after_minus_before": "Δ Wind",
    "swvl1__delta_after_minus_before": "Δ SWVL1",
    "swvl2__delta_after_minus_before": "Δ SWVL2",
    "swvl3__delta_after_minus_before": "Δ SWVL3",
    "swvl4__delta_after_minus_before": "Δ SWVL4",
}

TOPK = 4
freq_target = "frequency__delta_after_minus_before"  # select top-4 using this target

# -------------------------
# HELPERS
# -------------------------
def corr_r(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def linreg_ci_band(x, y, x_grid=None):
    """
    OLS y = b0 + b1*x, and 95% CI for the MEAN prediction.
    Returns x_grid, y_hat, y_lo, y_hi
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 3:
        raise ValueError("Need >=3 points")

    xbar = x.mean()
    ybar = y.mean()
    Sxx = np.sum((x - xbar) ** 2)
    if Sxx == 0:
        raise ValueError("Sxx=0")

    Sxy = np.sum((x - xbar) * (y - ybar))
    b1 = Sxy / Sxx
    b0 = ybar - b1 * xbar

    yhat = b0 + b1 * x
    resid = y - yhat
    s2 = np.sum(resid ** 2) / (n - 2)
    s = np.sqrt(s2)

    tcrit = 1.96 if n >= 30 else 2.0

    if x_grid is None:
        x_grid = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    x_grid = np.asarray(x_grid, dtype=float)

    y_grid = b0 + b1 * x_grid
    se_mean = s * np.sqrt(1.0 / n + (x_grid - xbar) ** 2 / Sxx)
    y_lo = y_grid - tcrit * se_mean
    y_hi = y_grid + tcrit * se_mean
    return x_grid, y_grid, y_lo, y_hi

# -------------------------
# RUN
# -------------------------
df = pd.read_csv(in_csv)

needed = targets + drivers
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

X = df[needed].apply(pd.to_numeric, errors="coerce")

# -------------------------
# Pick TOPK drivers based on |corr(driver, ΔFreq)|
# -------------------------
r_list = []
for xcol in drivers:
    r = corr_r(X[xcol].values, X[freq_target].values)
    if np.isfinite(r):
        r_list.append((xcol, r))

r_list.sort(key=lambda t: abs(t[1]), reverse=True)
top4_drivers = [x for x, _ in r_list[:TOPK]]

print("Top 4 drivers based on ΔFreq:", [short.get(d, d) for d in top4_drivers])

# -------------------------
# PLOT: 4 rows x 3 columns (same 4 drivers for all targets)
# -------------------------
fig = plt.figure(figsize=(12, 12))
gs = GridSpec(nrows=TOPK, ncols=len(targets), figure=fig, hspace=0.45, wspace=0.30)

for col, ycol in enumerate(targets):
    for row, xcol in enumerate(top4_drivers):
        ax = fig.add_subplot(gs[row, col])

        pair = X[[xcol, ycol]].dropna()
        x = pair[xcol].values
        y = pair[ycol].values

        ax.scatter(x, y, s=10, alpha=0.75)

        r = corr_r(x, y)
        if len(pair) >= 3 and np.isfinite(r):
            try:
                xg, yg, lo, hi = linreg_ci_band(x, y)
                ax.plot(xg, yg, linewidth=2)
                ax.fill_between(xg, lo, hi, alpha=0.18)
            except Exception:
                pass

        ax.set_title(f"{short.get(ycol, ycol)} vs {short.get(xcol, xcol)} (r={r:.2f})", fontsize=10)
        ax.set_xlabel(short.get(xcol, xcol))
        ax.set_ylabel(short.get(ycol, ycol))
        ax.grid(True, linewidth=0.4, alpha=0.35)

plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close()
print("Saved:", out_png)


Top 4 drivers based on ΔFreq: ['Δ VPD', 'Δ Tmax', 'Δ SWVL1', 'Δ SWVL2']
Saved: C:\Users\geoec\OneDrive\Desktop\Papers\Thirst_Wave\01_Data\top4_freq_drivers_applied_to_all_targets_3cols.png
